# 07 - significance testing, zero-shot comparison, feature weights, and figures

Runs the full analysis behind the benchmark document:
1. `run_analysis.py`: zero-shot AUROC per architecture, DeLong plus paired-bootstrap significance (best PLM vs identity floor, contiguous), and the all-features LR feature-weight ranking.
2. `build_architecture_comparison.py`: curated architecture x representation summary (a reconstructed generator, see its own docstring).
3. `make_benchmark_figs.py`: the five document figures.

Needs the arm tables from notebook 06 (`plm_benchmark_sequence_only.csv`, `plm_benchmark_esmfold.csv`).

In [ ]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'.projectroot').exists())
sys.path.insert(0, str(ROOT))
from paths import TABLES, FIGURES, DOCS
print("project root:", ROOT)

In [ ]:
import subprocess, sys, os
def run_in(env, script, *args):
    """Run a src/ script inside a named conda env, streaming output."""
    py = Path(sys.executable).parent.parent.parent / env / "bin" / "python"
    if not py.exists():  # fallback: same interpreter
        py = sys.executable
    cmd = [str(py), str(ROOT/"src"/script), *args]
    print("running:", " ".join(cmd))
    r = subprocess.run(cmd, cwd=str(ROOT/"notebooks"), capture_output=True, text=True)
    print(r.stdout[-3000:]); 
    if r.returncode!=0: print("STDERR:", r.stderr[-2000:]); raise SystemExit(r.returncode)

## 1. Significance, zero-shot, and feature weights
Recomputes contiguous OOF predictions for the top PLM cell and the identity floor, then runs DeLong and a 5,000-fold paired bootstrap. Also computes zero-shot AUROC and the all-features LR weights. About 4-5 min on CPU.

In [ ]:
run_in('betalactam-v2','run_analysis.py')
# -> results/tables/{zeroshot_vs_classifier,significance_plm_vs_floor,feature_weights_by_block,feature_weights_top20}.csv

## 1b. Architecture comparison

Reconstructs `results/tables/architecture_comparison.csv`, the curated per-architecture, per-representation summary `make_benchmark_doc.py` reads for the results document. This file previously had no generator anywhere in the pipeline. `build_architecture_comparison.py` reverse-engineers it from `plm_benchmark_sequence_only.csv`, and was checked to reproduce the existing file exactly before being wired in here.

In [ ]:
run_in('betalactam-v2', 'build_architecture_comparison.py')
# -> results/tables/architecture_comparison.csv

## 2. Figures
Renders the five publication figures from the result tables.

In [ ]:
run_in('betalactam-v2','make_benchmark_figs.py')
# -> results/figures/fig_{plm_vs_floor,architecture_comparison,zeroshot_vs_classifier,esmfold_arm,feature_weights}.png

## 3. Inspect the headline numbers

In [ ]:
import pandas as pd
sig = pd.read_csv(TABLES/'significance_plm_vs_floor.csv'); print('SIGNIFICANCE'); print(sig.to_string(index=False)); print()
zt  = pd.read_csv(TABLES/'zeroshot_vs_classifier.csv');   print('ZERO-SHOT vs CLASSIFIER'); print(zt.to_string(index=False)); print()
wb  = pd.read_csv(TABLES/'feature_weights_by_block.csv'); print('FEATURE WEIGHTS (per block)'); print(wb.to_string(index=False))